# Volunteer Hotspot Finder - Example Notebook

Find coverage gaps in eBird data to prioritize volunteer survey efforts.

## Setup

1. Copy `.env.example` to `.env` and add your eBird API key
2. Install: `pip install -e ..`

In [ ]:
from pathlib import Path
from birdlife_hotspot_finder import Config, HotspotFinder

## Initialize Finder

Load config from `.env` file:

In [ ]:
config = Config(_env_file=Path(".env"))
finder = HotspotFinder.from_config(config)
print(f"Grid size: {config.grid_size_km}km")
print(f"Cache type: {config.cache_type}")

## Find Coverage Gaps by Region

Supported regions: RS (Serbia), SE (Sweden), ES (Spain), CH (Switzerland), US-NY, US-CA, US-TX

In [ ]:
# Find top 5 coverage gaps in Serbia
result = await finder.find_gaps(region="RS", limit=5)

print(f"Found {len(result.data)} gaps")
print(f"Query timestamp: {result.meta.query_timestamp}")

In [ ]:
# Display gaps as table
for i, loc in enumerate(result.data, 1):
    ext = loc.extensions
    print(f"{i}. {loc.name}")
    print(f"   Coords: {loc.decimal_latitude}, {loc.decimal_longitude}")
    print(f"   Priority: {ext.get('coverage.priorityScore')}")
    print(f"   Nearest hotspot: {ext.get('coverage.nearestHotspotName')} ({ext.get('coverage.nearestHotspotDistanceKm')}km)")
    print()

## Find Gaps Near Coordinates

Search around a specific location:

In [ ]:
# Find gaps near Belgrade (44.8, 20.4)
nearby = await finder.find_gaps(lat=44.8, lng=20.4, radius_km=30, limit=5)

for loc in nearby.data:
    print(f"{loc.name} - {loc.extensions.get('coverage.nearestHotspotDistanceKm')}km to nearest hotspot")

## Export as JSON

Get API Response Convention format:

In [ ]:
import json

response = result.to_response_dict()
print(json.dumps(response, indent=2))

## Cleanup

In [ ]:
await finder.close()